# Script to update metadata for any dataset

Check metadata for old sequences to see if anything has been added

Databases: GISAID, Andersen, NCBI Virus

In [ ]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# Dates
start_date = "11-01-2021"
end_date = "06-05-2025"
date_range = start_date + "--" + end_date
update_date = "06-10-2025"

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads_gisaid = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder for gisaid
genotype = "B3.13"
genotype_underscored = genotype.replace(".", "_")
originals = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "_" + genotype_underscored + "/"


os.chdir(originals)

## Original Files

In [ ]:
original_fasta_dfs = {}

for dirpath, dirs, files in os.walk(originals):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name)
            original_fasta_dfs[file_name] = fasta_file

## GISAID

In [ ]:
# Get data from GISAID

username = input("Username: ")
password = input("Password: ")
browser = input("Browser: ")
sleep_time = input("Seconds to sleep in between clicks: ")
continent = input("Continent(s) separated by commas: ")
start_date = dateutil.parser.parse(start_date).strftime("%Y-%m-%d") # Make sure date is in correct format
end_date = dateutil.parser.parse(end_date).strftime("%Y-%m-%d")

open_gisaid(username, password, browser, sleep_time, continent, start_date, end_date)

In [ ]:
# Get downloaded GISAID data

all_metadata_files = []
all_fasta_files = []

# Grab files
for dirpath, dirs, files in os.walk(downloads_gisaid):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)

In [ ]:
# Search for dates and states based on isolate

for key in original_fasta_dfs:
    df = original_fasta_dfs[key]
    # isolate_ids = df["Isolate_Id"]
    

## Andersen

In [ ]:
# Read metadata

os.chdir(home)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = home + "Andersen/avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv") # Everything else

print(len(metadata))

# Merge with metadata_normalized
metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# # Find only >= last date using Release Date from metadata 
# metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# # Find only <= update date using Release Date from metadata
# metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) 
display(metadata)

In [ ]:
# Collapse dataset to only sequences without dates OR states



## NCBI Virus